# QLoRA Fine-Tuning with Unsloth: A Complete Tutorial

This notebook demonstrates how to fine-tune a Large Language Model (LLM) using **QLoRA** (Quantized Low-Rank Adaptation) with the Unsloth library on NVIDIA GPUs.

---

## What is QLoRA?

**QLoRA** extends the LoRA technique by adding 4-bit quantization to the base model weights, dramatically reducing memory usage while maintaining training quality.

| Technique | Base Model Precision | Adapter Precision | VRAM Usage |
|-----------|---------------------|-------------------|------------|
| Full Fine-Tuning | 16-bit | N/A | Very High |
| LoRA | 16-bit | 16-bit | Medium |
| **QLoRA** | **4-bit (NF4)** | 16-bit | **Low** |

### Key QLoRA Innovations:
1. **4-bit NormalFloat (NF4)**: A data type optimized for normally distributed weights
2. **Double Quantization**: Quantizes the quantization constants to save additional memory
3. **Paged Optimizers**: Manages memory spikes during gradient checkpointing

---

## Why Use Unsloth?

**Unsloth** provides optimized implementations that offer:
- **2x faster training** compared to standard implementations
- **Up to 70% less memory usage**
- **Zero accuracy degradation**
- Full compatibility with HuggingFace ecosystem

---

## Requirements

- NVIDIA GPU (GTX 1070+, RTX series, A100, H100)
- CUDA installed
- Python 3.8+

---

## 1. Environment Setup

First, we need to install Unsloth and its dependencies. This includes:

- **bitsandbytes**: Enables 4-bit quantization for QLoRA
- **accelerate**: HuggingFace's library for distributed training
- **xformers**: Memory-efficient attention implementations
- **peft**: Parameter-Efficient Fine-Tuning library
- **trl**: Transformer Reinforcement Learning, includes SFTTrainer
- **unsloth**: The optimization library we're using

In [ ]:
# Install Unsloth and required dependencies
# Note: --no-deps prevents dependency conflicts
# Uncomment if you have already installed the dependencies or if you are running this on Colab.

# %pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo matplotlib
# %pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
# %pip install --no-deps unsloth

### 1.1 Verify CUDA Availability

Before proceeding, let's verify that CUDA is available and check our GPU specifications. QLoRA requires a CUDA-enabled NVIDIA GPU.

In [ ]:
import torch
import logging

logging.basicConfig(level=logging.INFO)

# Check CUDA availability
logging.info(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    logging.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logging.info(
        f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    logging.info(f"CUDA Version: {torch.version.cuda}")
else:
    raise RuntimeError("CUDA is not available. QLoRA requires an NVIDIA GPU.")

---

## 2. Model Configuration with QLoRA Quantization

This is where QLoRA differs from standard LoRA. We'll configure **4-bit quantization** using `BitsAndBytesConfig`.

### QLoRA Quantization Parameters Explained:

| Parameter | Description | Recommended Value |
|-----------|-------------|------------------|
| `load_in_4bit` | Enable 4-bit quantization | `True` |
| `bnb_4bit_quant_type` | Quantization data type | `"nf4"` (NormalFloat4) |
| `bnb_4bit_compute_dtype` | Computation precision | `torch.bfloat16` |
| `bnb_4bit_use_double_quant` | Double quantization for extra memory savings | `True` |

### Understanding NF4 (NormalFloat4):
NF4 is an information-theoretically optimal data type for normally distributed weights. Since neural network weights typically follow a normal distribution, NF4 provides better representation than standard 4-bit integers.

In [ ]:
from unsloth import FastLanguageModel
import torch

# ============================================
# QLoRA Configuration
# ============================================

# Model to fine-tune (using Unsloth's pre-quantized 4-bit model for efficiency)
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

# Maximum sequence length for training
# Longer sequences = more context but more memory usage
MAX_SEQ_LENGTH = 2048

# Enable 4-bit quantization (core of QLoRA)
LOAD_IN_4BIT = True

# Auto-detect optimal dtype:
# - Float16 for older GPUs (T4, V100)
# - Bfloat16 for newer GPUs (Ampere+: A100, RTX 3090+)
DTYPE = None

logging.info("Loading model with QLoRA 4-bit quantization...")
logging.info(f"Model: {MODEL_NAME}")
logging.info(f"Max sequence length: {MAX_SEQ_LENGTH}")

### 2.1 Load the Model with Unsloth

Unsloth's `FastLanguageModel.from_pretrained()` automatically applies optimizations and returns both the model and tokenizer. When using a pre-quantized model (ending in `-bnb-4bit`), the 4-bit quantization is already applied.

In [ ]:
# Load the model with Unsloth optimizations
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,  # None = auto-detection
    load_in_4bit=LOAD_IN_4BIT,  # Enable QLoRA 4-bit quantization
)

logging.info("\n✓ Model loaded successfully with 4-bit quantization!")

---

## 3. LoRA Adapter Configuration

Now we add **LoRA adapters** to the quantized model. These small, trainable matrices are inserted into specific layers, allowing efficient fine-tuning.

### LoRA Parameters Explained:

| Parameter | Description | Impact |
|-----------|-------------|--------|
| `r` (rank) | Dimension of LoRA matrices | Higher = more capacity, more memory |
| `lora_alpha` | Scaling factor | Controls adapter influence strength |
| `lora_dropout` | Regularization dropout | Prevents overfitting (0 is optimized) |
| `target_modules` | Which layers to adapt | More modules = more flexibility |

### Target Modules:
We target all linear projections in both **attention** and **MLP** blocks:

- **Attention**: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- **MLP**: `gate_proj`, `up_proj`, `down_proj`

In [ ]:
# ============================================
# LoRA Adapter Configuration
# ============================================

# LoRA rank: Higher = more trainable parameters, more capacity
# Recommended: 8, 16, 32, 64 depending on task complexity
LORA_R = 16

# LoRA alpha: Scaling factor for the adapters
# Common practice: set equal to rank or 2x rank
LORA_ALPHA = 16

# Dropout: 0 is optimized by Unsloth for maximum speed
LORA_DROPOUT = 0

# Bias type: "none" is optimized and recommended
LORA_BIAS = "none"

# Target all linear layers for maximum adaptability
TARGET_MODULES = [
    "q_proj",     # Query projection (attention)
    "k_proj",     # Key projection (attention)
    "v_proj",     # Value projection (attention)
    "o_proj",     # Output projection (attention)
    "gate_proj",  # Gate projection (MLP)
    "up_proj",    # Up projection (MLP)
    "down_proj",  # Down projection (MLP)
]

### 3.1 Apply LoRA Adapters to the Model

We use `FastLanguageModel.get_peft_model()` to add LoRA adapters. The `use_gradient_checkpointing="unsloth"` option enables Unsloth's optimized gradient checkpointing, which reduces VRAM usage by 30% compared to standard implementations.

In [ ]:
# Apply LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias=LORA_BIAS,
    # Unsloth's optimized gradient checkpointing
    # Uses 30% less VRAM and supports 2x larger batch sizes
    use_gradient_checkpointing="unsloth",
    random_state=3407,  # For reproducibility
    use_rslora=False,   # Rank-Stabilized LoRA (optional)
    loftq_config=None,  # LoftQ initialization (optional)
)

logging.info("\n✓ LoRA adapters applied successfully!")

### 3.2 Analyze Trainable Parameters

One of the key benefits of QLoRA is that we only train a tiny fraction of the model's parameters. Let's see how many parameters are actually trainable.

In [ ]:
# Calculate trainable vs total parameters
trainable_params = sum(p.numel()
                       for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100 * trainable_params / all_params

logging.info("=" * 50)
logging.info("PARAMETER SUMMARY")
logging.info("=" * 50)
logging.info(f"Trainable parameters: {trainable_params:,}")
logging.info(f"Total parameters: {all_params:,}")
logging.info(f"Trainable percentage: {trainable_percent:.4f}%")
logging.info("=" * 50)

---

## 4. Dataset Preparation

For this tutorial, we'll use the **FineTome-100k** dataset, a high-quality instruction-following dataset. The dataset contains conversations in ShareGPT format.

### What is ShareGPT Format?
ShareGPT format structures conversations as a list of turns with `role` (user/assistant) and `content` fields:

```json
{
  "conversations": [
    {"role": "user", "content": "What is AI?"},
    {"role": "assistant", "content": "AI is..."}
  ]
}
```

In [ ]:
from datasets import load_dataset

# Load the FineTome-100k dataset
# This is a curated instruction-following dataset
dataset = load_dataset("mlabonne/FineTome-100k", split="train")

logging.info(f"\n✓ Dataset loaded successfully!")
logging.info(f"Number of examples: {len(dataset):,}")
logging.info(f"\nExample structure:")
logging.info(dataset[0])

---

## 5. Chat Template and Prompt Formatting

Each model family has its own **chat template** that structures conversations. We need to apply the correct template for Llama 3.1/3.2 models.

### Why Chat Templates Matter:
- Models are trained with specific delimiters and formatting
- Using the wrong template can significantly degrade performance  
- Unsloth provides `get_chat_template()` to easily apply the correct format

In [ ]:
from typing import Any, cast
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

# Apply the Llama 3.1 chat template to the tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",  # Use Llama 3.1/3.2 template
)

# Standardize dataset to ShareGPT format
dataset = standardize_sharegpt(dataset)

logging.info("✓ Chat template applied: llama-3.1")

### 5.1 Create the Formatting Function

We need a function that converts each conversation into the model's expected format. This function will be applied to the entire dataset.

In [ ]:
def formatting_prompts_func(examples):
    """
    Convert conversations to the model's expected text format.

    Args:
        examples: Batch of examples from the dataset

    Returns:
        Dictionary with 'text' field containing formatted prompts
    """
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,  # Return string, not tokens
            add_generation_prompt=False  # Don't add prompt for next response
        )
        for convo in convos
    ]
    return {"text": texts}


# Apply formatting to the dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

logging.info("✓ Dataset formatted successfully!")

### 5.2 Inspect a Formatted Example

Let's look at how a conversation looks before and after formatting to understand the transformation.

In [ ]:
# View an example
item = cast(Any, dataset[5])

logging.info("=" * 50)
logging.info("ORIGINAL CONVERSATION FORMAT:")
logging.info("=" * 50)
logging.info(item["conversations"])

logging.info("\n" + "=" * 50)
logging.info("FORMATTED TEXT (Model Input):")
logging.info("=" * 50)
logging.info(item["text"])

---

## 6. Training Configuration

Now we configure the training parameters. These settings are optimized for QLoRA training with limited GPU memory.

### Key Training Parameters:

| Parameter | Description | Memory Impact |
|-----------|-------------|---------------|
| `per_device_train_batch_size` | Samples per GPU per step | ↑ = more memory |
| `gradient_accumulation_steps` | Steps before weight update | No extra memory |
| `learning_rate` | How fast the model learns | None |
| `optim="adamw_8bit"` | 8-bit optimizer | Reduces memory |

### Effective Batch Size Formula:
```
Effective Batch Size = per_device_batch_size × gradient_accumulation_steps × num_gpus
```

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

# ============================================
# Training Configuration
# ============================================

OUTPUT_DIR = "./outputs_qlora"  # Where to save checkpoints

# Batch size settings
PER_DEVICE_TRAIN_BATCH_SIZE = 2    # Samples per GPU per step
GRADIENT_ACCUMULATION_STEPS = 4    # Accumulate gradients over 4 steps
# Effective batch size = 2 * 4 = 8

# Learning settings
LEARNING_RATE = 2e-4               # Standard for LoRA/QLoRA
WARMUP_STEPS = 10                  # Gradual learning rate increase
MAX_STEPS = 60                     # Total training steps (-1 = use epochs)

# Logging
LOGGING_STEPS = 10                 # Log every N steps

# Optimizer: 8-bit AdamW saves memory
OPTIMIZER = "adamw_8bit"

# Auto-detect precision support
use_bf16 = is_bfloat16_supported()
use_fp16 = not use_bf16

logging.info(f"Precision: {'bfloat16' if use_bf16 else 'float16'}")
logging.info(
    f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

### 6.1 Create Training Arguments

We use HuggingFace's `TrainingArguments` to configure all training parameters.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batch settings
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Training duration
    # num_train_epochs=3,  # Uncomment for full training
    max_steps=MAX_STEPS,   # For demo, limit to 60 steps

    # Learning settings
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",

    # Precision
    fp16=use_fp16,
    bf16=use_bf16,

    # Optimizer
    optim=OPTIMIZER,
    weight_decay=0.01,

    # Logging
    logging_steps=LOGGING_STEPS,
    report_to="none",  # Set to "wandb" for Weights & Biases tracking

    # Reproducibility
    seed=3407,
)

logging.info("✓ Training arguments configured!")

---

## 7. Initialize the Trainer

We use TRL's `SFTTrainer` (Supervised Fine-Tuning Trainer), which is designed specifically for instruction-tuning language models.

### Important SFTTrainer Parameters:

- `dataset_text_field`: Which field contains the formatted text
- `max_seq_length`: Maximum sequence length for training
- `packing`: Whether to pack multiple short sequences into one (faster but may affect quality)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    tokenizer=tokenizer,

    # Dataset configuration
    dataset_text_field="text",     # Field containing formatted text
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,            # Parallel data processing

    # Packing: Can speed up training 5x for short sequences
    # Set to True for production, False for debugging
    packing=False,

    # Training arguments
    args=training_args,
)

logging.info("✓ SFTTrainer initialized!")

---

## 8. Train Only on Assistant Responses

An important optimization: we only compute loss on the **assistant's responses**, not on the user's questions. This focuses the model on learning to generate good responses.

### Why This Matters:
- User inputs are just context, we don't need to predict them
- Reduces noise in the training signal
- Makes training more efficient

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Configure training to only compute loss on assistant responses
trainer = train_on_responses_only(
    trainer,
    # These markers identify the start of user and assistant turns
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

logging.info("✓ Trainer configured to train on responses only!")

---

## 9. Start Training

Now we begin the fine-tuning process. The training progress will show:
- **Step**: Current training step
- **Training Loss**: How well the model is learning (lower is better)

**Note**: For this demo, we're only training for 60 steps. For production, increase `max_steps` or use `num_train_epochs`.

In [ ]:
logging.info("Starting QLoRA fine-tuning...")
logging.info("=" * 50)

# Begin training
trainer_stats = trainer.train()

logging.info("\n" + "=" * 50)
logging.info("✓ Training completed!")
logging.info(
    f"Total training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

In [ ]:
import matplotlib.pyplot as plt

# Extract loss history from trainer state
history = trainer.state.log_history

# Filter for loss values
loss_values = [x['loss'] for x in history if 'loss' in x]
steps = [x['step'] for x in history if 'loss' in x]

if len(loss_values) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(steps, loss_values, marker='o', linestyle='-',
             color='b', label='Training Loss')
    plt.title('QLoRA Training Loss Evolution')
    plt.xlabel('Training Steps')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No loss history found.")

### 9.1 Visualize Training Loss

Let's visualize how the model's loss decreased during training. A downward trend indicates the model is learning effectively.

---

## 10. Inference: Test Your Fine-Tuned Model

Let's test the fine-tuned model by generating a response. We use `FastLanguageModel.for_inference()` to enable Unsloth's optimized inference mode, which is 2x faster than standard inference.

In [ ]:
from unsloth.chat_templates import get_chat_template

# Prepare tokenizer for inference
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

# Enable optimized inference mode (2x faster)
FastLanguageModel.for_inference(model)

# Create a test message
messages = [
    {"role": "user", "content": "Explain what QLoRA is and why it's useful for fine-tuning LLMs."},
]

# Convert to model input format
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Add prompt for model to continue
    return_tensors="pt",
).to("cuda")

logging.info("Generating response...")

### 10.1 Generate and Display Response

In [ ]:
# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,    # Maximum length of generated response
    use_cache=True,        # Cache for faster generation
    temperature=0.7,       # Creativity (0=deterministic, 1=creative)
    min_p=0.1,             # Minimum probability threshold
)

# Decode the response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

logging.info("=" * 50)
logging.info("MODEL RESPONSE:")
logging.info("=" * 50)
logging.info(response)

---

## 11. Save the Fine-Tuned Model

After training, we need to save our model. There are several options:

1. **LoRA Adapters Only**: Small files (~100MB), requires base model for inference
2. **Merged Model**: Full model with adapters merged in
3. **GGUF Format**: For use with llama.cpp, Ollama, etc.

### 11.1 Save LoRA Adapters (Recommended)

In [ ]:
# Save directory for the adapters
SAVE_NAME = "Llama32_QLoRA_fine_tuned"

# Save the LoRA adapters and tokenizer
model.save_pretrained(SAVE_NAME)
tokenizer.save_pretrained(SAVE_NAME)

logging.info(f"✓ Model saved to: {SAVE_NAME}/")
logging.info("\nSaved files:")
!ls -la {SAVE_NAME}/

### 11.2 Export to GGUF Format (Optional)

GGUF is a format for running models with llama.cpp, Ollama, and other efficient inference engines. Uncomment the cell below to export.

In [ ]:
# Uncomment to export to GGUF format
# model.push_to_hub_gguf(
#     SAVE_NAME,
#     tokenizer,
#     quantization_method="q4_k_m"  # Good balance of size vs quality
# )

---

## 12. Load and Use the Fine-Tuned Model

Here's how to load your fine-tuned model for future inference sessions.

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

# Load the fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SAVE_NAME,  # Your saved model directory
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

# Enable optimized inference
FastLanguageModel.for_inference(model)

logging.info("✓ Fine-tuned model loaded successfully!")

### 12.1 Streaming Inference

For a better user experience, we can stream the response token by token using `TextStreamer`.

In [ ]:
# Create a new test message
messages = [
    {"role": "user", "content": "What are the benefits of parameter-efficient fine-tuning?"},
]

# Prepare inputs
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Create streamer for real-time output
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

logging.info("Streaming response:")
logging.info("-" * 50)

# Generate with streaming
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    min_p=0.1,
)

In [ ]:
import time

# Prepare a longer prompt for benchmarking
messages = [
    {"role": "user", "content": "Explain the theory of relativity in simple terms."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Warmup run
print("Warming up GPU...")
_ = model.generate(input_ids=inputs, max_new_tokens=20)

# Benchmark run
print("Running benchmark...")
torch.cuda.synchronize()
start_time = time.time()

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,    # Generate a substantial amount of text
    use_cache=True,
    temperature=0.7,
    min_p=0.1,
)

torch.cuda.synchronize()
end_time = time.time()

# Calculate metrics
generated_tokens = outputs.shape[1] - inputs.shape[1]
duration = end_time - start_time
tokens_per_sec = generated_tokens / duration

print(f"\nBENCHMARK RESULTS:")
print(f"Total time: {duration:.2f} seconds")
print(f"Tokens generated: {generated_tokens}")
print(f"Speed: {tokens_per_sec:.2f} tokens/sec")

### 12.2 Inference Speed Benchmark

Let's measure the generation speed (tokens per second) of our fine-tuned model. This is a critical metric for production deployment.

---

## 13. GPU Memory Analysis

Let's analyze the GPU memory usage during our QLoRA training to see how efficient it was.

In [ ]:
if torch.cuda.is_available():
    logging.info("=" * 50)
    logging.info("GPU MEMORY STATISTICS")
    logging.info("=" * 50)

    # Get memory statistics
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    max_allocated = torch.cuda.max_memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    logging.info(f"\nCurrent Memory Usage:")
    logging.info(f"  • Allocated: {allocated:.2f} GB")
    logging.info(f"  • Reserved:  {reserved:.2f} GB")
    logging.info(f"  • Peak:      {max_allocated:.2f} GB")
    logging.info(f"  • Total GPU: {total:.2f} GB")
    logging.info(f"  • Used:      {(max_allocated/total)*100:.1f}%")

    # Clear cache
    torch.cuda.empty_cache()
    logging.info("\n✓ GPU cache cleared!")
else:
    logging.info("CUDA not available for memory analysis.")

---

## 🎉 Congratulations!

You have successfully completed QLoRA fine-tuning with Unsloth! Here's what you learned:

### Key Takeaways:

1. **QLoRA = LoRA + 4-bit Quantization**: Uses NF4 quantization for dramatic memory savings
2. **Unsloth Optimizations**: 2x faster training, 70% less memory
3. **Parameter Efficiency**: Only ~1% of parameters are trained
4. **Train on Responses Only**: More efficient learning signal

### Next Steps:

- Try different `lora_r` values (8, 32, 64) to see the impact on quality
- Experiment with different datasets for your specific use case
- Export to GGUF for deployment with Ollama or llama.cpp
- Push to HuggingFace Hub to share your model

### Resources:

- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [QLoRA Paper](https://arxiv.org/abs/2305.14314)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [HuggingFace PEFT](https://huggingface.co/docs/peft)